# 第 1 周 · 第 1 天作业 —— 由主题行生成邮件

## 练习目标（理念）

根据给定的 **邮件主题行（subject line）** 字符串，用 OpenAI Chat Completions 生成一封格式化邮件正文。

你会练习到：

1. 自定义 **system prompt** 与 **user prompt 前缀**
2. 按 API 要求组装 `messages`（`system` / `user`）
3. 用模型根据 `email_subject` 生成邮件
4. 把模型返回的 Markdown 在笔记本里展示出来

## 和本课 Day 1 的关系

| 本课概念 | 本作业里你会看到 |
|----------|------------------|
| `load_dotenv` + `OPENAI_API_KEY` | 从 `.env` 读密钥，不写进代码 |
| Chat Completions | `openai.chat.completions.create(...)` |
| `messages` | system 定角色，user = 前缀 + 主题行 |
| Markdown 展示 | `display(Markdown(...))` |

## 怎么跑

1. 准备 `.env`，写入有效的 `OPENAI_API_KEY`
2. 从上到下运行单元格；可改 `email_subject` 再跑一次，对比不同主题下的生成效果


In [ ]:
# ========== 第 1 步：导入库、加载环境、写好提示词 ==========

# 导入标准库 os：读环境变量（Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown / display：在笔记本里渲染模型输出
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI

# 加载 .env：override=True 表示文件值覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 OPENAI_API_KEY（本格后面创建客户端时，SDK 也会自行读取该变量）
api_key = os.getenv('OPENAI_API_KEY')

# system prompt：定模型角色——擅长根据主题行写邮件；保留英文以免改变行为
system_prompt = "You are an expert at generating email contents from subject lines"
# user prompt 前缀：说明期望输出格式（要点列表等）；真正主题行会拼在后面
user_prompt_prefix = """
    Generate a formatted email message with bullet lists
    and other details given the subject line provided:

"""

# ========== 第 2 步：用 email_subject 组装 OpenAI messages 列表 ==========

# 入参是主题行字符串；返回 Chat Completions 需要的 messages
def messages_for(email_subject):
    return [
        # system：角色与能力边界
        {"role": "system", "content": system_prompt},
        # user：前缀说明 + 具体主题行（拼接后整段作为 user content）
        {"role": "user", "content": user_prompt_prefix + email_subject}
    ]

# 本次要生成邮件的主题行（改这里就能换场景）
email_subject = "Random Shapes Inc. LLM Engineer position application - Salvador Pasquier"
# 调用上面的函数，得到发给 API 的 messages
messages = messages_for(email_subject)

# ========== 第 3 步：调用 OpenAI Chat Completions（模型 gpt-5-nano） ==========

# 创建客户端：默认从环境变量取 OPENAI_API_KEY
openai = OpenAI()
# 非流式请求：把 model id 与 messages 交给 API，等整段生成完
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
# 从返回结构里取出助手正文（通常是 Markdown 文本）
email_message = response.choices[0].message.content

# ========== 第 4 步：以 Markdown 在笔记本中显示结果 ==========
display(Markdown(email_message))
